# 3. Parsing Custom Data & Configuration

So far we've used built-in datasets, which already have standard `t`/`x`/`y`/`pupil` columns. This notebook covers
two things you need to bring your **own** recordings into pEYES:
1. `peyes.parse_data` — renaming/cleaning your columns into that standard shape.
2. Global configuration (`peyes.set_screen_monitor`, `peyes.set_event_configurations`, `peyes.set_viewer_distance`)
   — package-wide defaults such as screen geometry and per-event-type duration bounds.

## Parsing your own data

Suppose your eye tracker exports columns named differently, and marks missing samples with a sentinel value (e.g.
`-1`) instead of `NaN`. `peyes.parse_data` renames the columns you tell it about and converts missing-data sentinels
to `NaN`:

In [1]:
import numpy as np
import pandas as pd
import peyes
import _helpers

d = _helpers.load_example_trial()
raw = pd.DataFrame({"time": d["t"], "X": d["x"], "Y": d["y"], "Pupil": d["pupil"]})
raw.loc[5:7, ["X", "Y"]] = -1  # simulate a tracker that marks missing samples as -1

parsed = peyes.parse_data(raw, time_name="time", x_name="X", y_name="Y", pupil_name="Pupil", missing_data_value=-1)
parsed.columns.tolist(), parsed.iloc[4:9]

(['t', 'x', 'y', 'pupil'],
         t         x         y  pupil
 4   8.002  521.3199  422.2191    NaN
 5  10.009       NaN       NaN    NaN
 6  12.007       NaN       NaN    NaN
 7  14.006       NaN       NaN    NaN
 8  16.001  522.8697  422.9716    NaN)

`parsed` now has the standard `t`, `x`, `y`, `pupil` columns pEYES expects everywhere else in this guide — the
`-1` sentinels became `NaN`. From here, `parsed["t"].values` etc. feed into `create_detector(...).detect(...)`
exactly like the built-in datasets did.

## Global configuration

Three functions set process-wide defaults, used whenever a value isn't given explicitly:
- `peyes.set_viewer_distance(distance_cm)` — default distance from screen to eyes.
- `peyes.set_screen_monitor(width_cm, height_cm, width_px, height_px)` — screen size and resolution, which also
  determines pixel size and screen bounds.
- `peyes.set_event_configurations(event_type, min_duration, max_duration, hex_color)` — per-event-type duration
  bounds (used to flag outlier events) and plot color.

These matter most for `Event.is_outlier` / `Event.get_outlier_reasons()`, which check an event's duration and pixel
coordinates against these configured bounds. Let's see it react live. First, a normal (non-outlier) fixation:

In [2]:
detector = peyes.create_detector("engbert", missing_value=np.nan, min_event_duration=4, pad_blinks_time=0)
labels, _ = detector.detect(
    t=d["t"], x=d["x"], y=d["y"], pixel_size_cm=d["pixel_size"], viewer_distance_cm=d["viewer_distance"],
)
events = peyes.create_events(
    labels=labels, t=d["t"], x=d["x"], y=d["y"], pupil=d["pupil"],
    pixel_size=d["pixel_size"], viewer_distance=d["viewer_distance"],
)
fixation = next(e for e in events if e.label == peyes.parse_label("fixation"))
print(f"duration={fixation.duration:.1f}ms  is_outlier={fixation.is_outlier}  reasons={fixation.get_outlier_reasons()}")

duration=204.0ms  is_outlier=False  reasons=[]


Now tighten the minimum fixation duration above this event's actual duration — the same event is immediately
flagged as an outlier, with no change to the event itself:

In [3]:
peyes.set_event_configurations("fixation", min_duration=fixation.duration + 10)
print(f"is_outlier={fixation.is_outlier}  reasons={fixation.get_outlier_reasons()}")

peyes.set_event_configurations("fixation", min_duration=4)  # restore the default used elsewhere in this guide

is_outlier=True  reasons=['min_duration']


`set_screen_monitor` works the same way for pixel-coordinate bounds — shrinking the configured screen
resolution below the event's actual pixel coordinates flags it as off-screen:

In [4]:
peyes.set_screen_monitor(width_px=10, height_px=10)
print(f"is_outlier={fixation.is_outlier}  reasons={fixation.get_outlier_reasons()}")

peyes.set_screen_monitor(width_px=1920, height_px=1080)  # restore

is_outlier=True  reasons=['pixel_outside_screen']


`set_viewer_distance(distance_cm)` sets the default viewer distance the same way, for whenever a function
accepts an optional viewer distance and none is given explicitly. In this guide we always pass `viewer_distance`
explicitly (as returned by the dataset or your own recording setup), so we won't stage a separate demo for it —
call it once at the start of your own analysis if your setup has one fixed viewing distance.

## What's next

**[4 Detection Algorithms](./4%20Detection%20Algorithms.ipynb)** — the other detection algorithms besides Engbert,
and how to choose between them.